# Adaptive V4 + V5 Evaluation and V6 Replay

This notebook does not train models. It mounts the existing Drive datasets, restores the downloaded V4/V5 checkpoints into Colab temporary disk, evaluates V4 and V5 on the same validation protocol, then runs a bounded V6 point-in-time replay. Large checkpoints and caches stay in `/content`; only lightweight reports are downloaded.

The current Drive audit found complete `train` and `validation` adaptive splits. The incomplete `test` split is intentionally blocked.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

from pathlib import Path
import gc, hashlib, importlib.util, json, shutil, subprocess, sys, zipfile

DRIVE_ROOT = Path('/content/drive/MyDrive/recsys')
ADAPTIVE_ROOT = DRIVE_ROOT / 'data/gold/adaptive/v1'
BASE_GOLD_ROOT = DRIVE_ROOT / 'data/gold/two_tower/v1_colab'
WORK_ROOT = Path('/content/recsys_adaptive_eval')
SOURCE_ROOT = WORK_ROOT / 'runtime'
UPLOAD_ROOT = WORK_ROOT / 'uploads'
V4_DIR = WORK_ROOT / 'artifacts/v4_sequence_retrieval'
V5_DIR = WORK_ROOT / 'artifacts/v5_adaptive_ranker'
EVAL_DIR = WORK_ROOT / 'reports/evaluation'
V6_DIR = WORK_ROOT / 'reports/v6_online_replay'
for path in [SOURCE_ROOT, UPLOAD_ROOT, V4_DIR, V5_DIR, EVAL_DIR, V6_DIR]:
    path.mkdir(parents=True, exist_ok=True)
print('adaptive:', ADAPTIVE_ROOT)
print('base gold:', BASE_GOLD_ROOT)

In [ ]:
# Install only missing packages; Colab already provides torch/numpy/pandas in normal GPU runtimes.
required = {'pyarrow': 'pyarrow', 'pandas': 'pandas', 'torch': 'torch'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('dependencies ready')

In [ ]:
# Audit required Drive inputs before loading the 3.3 GB checkpoint.
def parquet_count(path: Path) -> int:
    return sum(1 for file in path.rglob('*.parquet') if not file.name.startswith('.')) if path.exists() else 0

required_paths = {
    'adaptive_train': ADAPTIVE_ROOT / 'sequence_examples/train',
    'adaptive_validation': ADAPTIVE_ROOT / 'sequence_examples/validation',
    'numeric_stats': BASE_GOLD_ROOT / 'transforms/numeric_stats.json',
    'user_active_vocab': BASE_GOLD_ROOT / 'vocabularies/user_active_degree',
    'video_type_vocab': BASE_GOLD_ROOT / 'vocabularies/video_type',
    'upload_type_vocab': BASE_GOLD_ROOT / 'vocabularies/upload_type',
}
for name, path in required_paths.items():
    print(f'{name:24s}', 'OK' if path.exists() else 'MISSING', path)
missing = [str(path) for path in required_paths.values() if not path.exists()]
assert not missing, 'Missing Drive inputs:\n' + '\n'.join(missing)
for split in ['train', 'validation', 'test']:
    path = ADAPTIVE_ROOT / 'sequence_examples' / split
    count = parquet_count(path)
    success = (path / '_SUCCESS').exists()
    print(f'{split:10s}: parquet={count}, _SUCCESS={success}')
assert parquet_count(required_paths['adaptive_train']) > 0
assert parquet_count(required_paths['adaptive_validation']) > 0
assert (required_paths['adaptive_train'] / '_SUCCESS').exists()
assert (required_paths['adaptive_validation'] / '_SUCCESS').exists()
print('Validation protocol inputs are complete. Test remains intentionally unused.')

## Load the small evaluation runtime

Upload `adaptive_eval_runtime.zip` from the project root. This contains only the model definitions and evaluation scripts; it does not contain data or checkpoints.

In [ ]:
uploaded = files.upload()
runtime_names = [name for name in uploaded if Path(name).name.startswith('adaptive_eval_runtime') and name.endswith('.zip')]
assert runtime_names, 'Select adaptive_eval_runtime.zip'
runtime_zip = WORK_ROOT / 'adaptive_eval_runtime.zip'
runtime_zip.write_bytes(uploaded[runtime_names[0]])
del uploaded
gc.collect()
with zipfile.ZipFile(runtime_zip) as archive:
    archive.extractall(SOURCE_ROOT)
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
print('runtime ready:', SOURCE_ROOT)

## Upload V4 checkpoint parts

Upload the 27 files `best_model.pt.part_0000` through `best_model.pt.part_0026` plus the V4 `manifest.json`. Use the four cells below in batches to keep browser uploads stable. The final restore cell checks every part and the complete SHA256.

In [ ]:
V4_UPLOAD = UPLOAD_ROOT / 'v4'
V5_UPLOAD = UPLOAD_ROOT / 'v5'
V4_UPLOAD.mkdir(parents=True, exist_ok=True)
V5_UPLOAD.mkdir(parents=True, exist_ok=True)

def upload_batch(destination: Path, label: str):
    print(label)
    uploaded = files.upload()
    for name, payload in uploaded.items():
        (destination / Path(name).name).write_bytes(payload)
    names = sorted(uploaded)
    del uploaded
    gc.collect()
    print('saved', len(names), 'files:', names[0] if names else '-', '...', names[-1] if names else '-')

upload_batch(V4_UPLOAD, 'Select V4 manifest.json and parts 0000-0006')

In [ ]:
upload_batch(V4_UPLOAD, 'Select V4 parts 0007-0013')

In [ ]:
upload_batch(V4_UPLOAD, 'Select V4 parts 0014-0020')

In [ ]:
upload_batch(V4_UPLOAD, 'Select V4 parts 0021-0026')

## Upload V5 checkpoint

Upload either the complete V5 `best_model.pt`, or its one part plus V5 `manifest.json`.

In [ ]:
upload_batch(V5_UPLOAD, 'Select V5 best_model.pt OR V5 manifest.json + best_model.pt.part_0000')

In [ ]:
# Restore and cryptographically verify checkpoint parts.
def sha256(path: Path, block_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while block := stream.read(block_size):
            digest.update(block)
    return digest.hexdigest()

def restore_checkpoint(upload_dir: Path, output_dir: Path) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    direct_candidates = [path for path in upload_dir.glob('best_model*.pt') if '.part_' not in path.name]
    direct = direct_candidates[0] if direct_candidates else upload_dir / 'best_model.pt'
    target = output_dir / 'best_model.pt'
    manifests = sorted(upload_dir.glob('*manifest*.json')) + sorted(upload_dir.glob('manifest.json'))
    if direct.exists():
        shutil.copy2(direct, target)
        if manifests:
            manifest = json.loads(manifests[0].read_text())
            assert sha256(target) == manifest['full_sha256']
        print('restored direct checkpoint:', target, target.stat().st_size)
        return target
    assert manifests, f'No manifest found under {upload_dir}'
    manifest = json.loads(manifests[0].read_text())
    expected = [part['filename'] for part in manifest['parts']]
    # Colab adds suffixes such as ` (2)` when the browser has uploaded a duplicate name.
    for expected_name in expected:
        expected_path = upload_dir / expected_name
        if expected_path.exists():
            continue
        marker = '.part_'
        stem, part_number = expected_name.split(marker, 1)
        candidates = sorted(upload_dir.glob(f'{stem}*{marker}{part_number}'))
        assert candidates, f'Missing checkpoint part: {expected_name}'
        shutil.copy2(candidates[-1], expected_path)
        print('normalized', candidates[-1].name, '->', expected_name)
    missing = [name for name in expected if not (upload_dir / name).exists()]
    assert not missing, 'Missing checkpoint parts: ' + ', '.join(missing)
    with target.open('wb') as output:
        for part in manifest['parts']:
            source = upload_dir / part['filename']
            assert source.stat().st_size == part['size_bytes'], f'Bad size: {source.name}'
            assert sha256(source) == part['sha256'], f'Bad SHA256: {source.name}'
            with source.open('rb') as stream:
                shutil.copyfileobj(stream, output, length=8 * 1024 * 1024)
    assert target.stat().st_size == manifest['total_size_bytes']
    assert sha256(target) == manifest['full_sha256'], 'Full checkpoint SHA256 mismatch'
    print('restored verified checkpoint:', target, target.stat().st_size)
    return target

restore_checkpoint(V4_UPLOAD, V4_DIR)
restore_checkpoint(V5_UPLOAD, V5_DIR)

## Controlled V4 versus V5 evaluation

Protocol: latest strong-positive query per user, complete validation item pool, exact matrix retrieval, V5 reranking of exactly the same V4 Top-1000 candidates.

In [ ]:
device = 'cuda' if importlib.util.find_spec('torch') and __import__('torch').cuda.is_available() else 'cpu'
assert device == 'cuda', 'Use a Colab GPU runtime for full candidate evaluation.'
evaluate_script = SOURCE_ROOT / 'scripts/evaluate_adaptive_v4_v5.py'
command = [
    sys.executable, str(evaluate_script),
    '--adaptive-root', str(ADAPTIVE_ROOT),
    '--base-gold-root', str(BASE_GOLD_ROOT),
    '--v4-dir', str(V4_DIR),
    '--v5-dir', str(V5_DIR),
    '--output-dir', str(EVAL_DIR),
    '--split', 'validation',
    '--max-queries', '0',
    '--max-candidate-items', '0',
    '--retrieval-candidates', '1000',
    '--top-ks', '10,50,100',
    '--popularity-blend-grid', '0.0,0.02,0.05,0.08',
    '--item-batch-size', '8192',
    '--query-batch-size', '32',
    '--device', device,
]
print('running controlled validation evaluation...')
subprocess.run(command, check=True)
evaluation = json.loads((EVAL_DIR / 'adaptive_v4_v5_evaluation.json').read_text())
evaluation

## V6 bounded point-in-time replay

This evaluates timestamp-ordered point-in-time examples. Every sequence is sanitized to contain only events strictly earlier than its query. It is a production-oriented offline replay, not a live state-store service.

In [ ]:
# Historical V3 candidate-validation reference from the prior controlled run.
v3_reference = {
    'protocol_note': 'Historical validation candidate protocol; verify protocol fields before claiming a strict delta.',
    'hitrate@10': 0.24390243902439024,
    'hitrate@50': 0.5355691056910569,
    'hitrate@100': 0.6605691056910569,
    'ndcg@10': 0.03514699327560926,
    'ndcg@50': 0.030595039073765495,
    'ndcg@100': 0.028137996031824885,
}
v3_path = WORK_ROOT / 'v3_reference.json'
v3_path.write_text(json.dumps(v3_reference, indent=2))

replay_script = SOURCE_ROOT / 'scripts/run_adaptive_v6_replay.py'
command = [
    sys.executable, str(replay_script),
    '--adaptive-root', str(ADAPTIVE_ROOT),
    '--base-gold-root', str(BASE_GOLD_ROOT),
    '--v4-dir', str(V4_DIR),
    '--v5-dir', str(V5_DIR),
    '--v3-metrics', str(v3_path),
    '--output-dir', str(V6_DIR),
    '--split', 'validation',
    '--max-replay-queries', '1000',
    '--max-replay-users', '100',
    '--max-candidate-items', '0',
    '--retrieval-candidates', '1000',
    '--top-ks', '10,50,100',
    '--item-batch-size', '8192',
    '--query-batch-size', '32',
    '--max-report-rows', '1000',
    '--device', device,
]
print('running bounded V6 replay...')
subprocess.run(command, check=True)
comparison = json.loads((V6_DIR / 'online_model_comparison.json').read_text())
comparison

## Download lightweight reports

This archive contains metrics and CSV summaries only. It does not duplicate checkpoints or candidate caches.

In [ ]:
report_zip = Path('/content/adaptive_v4_v5_v6_reports.zip')
if report_zip.exists():
    report_zip.unlink()
with zipfile.ZipFile(report_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for report_dir in [EVAL_DIR, V6_DIR]:
        for path in report_dir.rglob('*'):
            if path.is_file():
                archive.write(path, path.relative_to(WORK_ROOT))
print('report archive:', report_zip, report_zip.stat().st_size, 'bytes')
files.download(str(report_zip))